# xformers --> memory efficient attention kernels

In [1]:
!pip install torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu128
!pip install unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

Looking in indexes: https://download.pytorch.org/whl/cu128
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 63.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.8/54.8 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 119.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 402.9/402.9 kB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 108.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 102.0 MB/s

In [4]:
import torch
import random
import numpy as np

In [8]:
# setting up seed for python, numpy, PyTorch, and for GPU's for reproducibility of random weights init and all
SEED = 3407
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

torch.cuda.manual_seed_all(SEED)

In [9]:
# all this code is abt performance optimization and memory optimization
torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision('high')


max_seq_len = 4096 #attention span of my model
dtype= None #placeholde, letting the system optimally detect
load_in_4bit = True #quantization (qLoRA)

In [10]:
assert torch.cuda.is_available()

In [11]:
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

In [12]:
base_model_name = 'unsloth/tinyllama-bnb-4bit'
model, tokenizer  = FastLanguageModel.from_pretrained(
    model_name=base_model_name,
    max_seq_length=max_seq_len,
    dtype = dtype,
    load_in_4bit=True
)

==((====))==  Unsloth 2026.3.15: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


But with kaiokendev's RoPE scaling of 2.0, it can be magically be extended to 4096!
<string>:45: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.


model.safetensors:   0%|          | 0.00/762M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/948 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/438 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [13]:
model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048, padding_idx=0)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=5632, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=5632, bias=False)
          (down_proj): Linear4bit(in_features=5632, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,)

In [14]:
tokenizer

LlamaTokenizerFast(name_or_path='unsloth/tinyllama-bnb-4bit', vocab_size=32000, model_max_length=4096, is_fast=True, padding_side='left', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<unk>'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [18]:
model = FastLanguageModel.get_peft_model(
    model,
     r= 32, #lora rank (8/16/32/64)
    target_modules = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj"
],
    lora_alpha = 64, #usually r or 2*r
    lora_dropout=0.0, #unsloth recommends 0
    bias = 'none', #good practice
    use_gradient_checkpointing=False, #set tRUE if model > 7B
    random_state = 3407
)

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
Unsloth 2026.3.15 patched 22 layers with 22 QKV layers, 22 O layers and 22 MLP layers.


In [19]:
model.print_trainable_parameters()

trainable params: 25,231,360 || all params: 1,125,279,744 || trainable%: 2.2422


In [21]:
model.peft_config

{'default': LoraConfig(task_type=<TaskType.CAUSAL_LM: 'CAUSAL_LM'>, peft_type=<PeftType.LORA: 'LORA'>, auto_mapping={'base_model_class': 'LlamaForCausalLM', 'parent_library': 'transformers.models.llama.modeling_llama', 'unsloth_fixed': True}, peft_version='0.18.1', base_model_name_or_path='unsloth/tinyllama-bnb-4bit', revision=None, inference_mode=False, r=32, target_modules={'down_proj', 'q_proj', 'v_proj', 'up_proj', 'k_proj', 'o_proj', 'gate_proj'}, exclude_modules=None, lora_alpha=64, lora_dropout=0.0, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, use_dora=False, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False)

In [22]:
import torch
torch.cuda.memory_summary

<function torch.cuda.memory.memory_summary(device: 'Device' = None, abbreviated: bool = False) -> str>

In [23]:
eos_token = tokenizer.eos_token

In [25]:
alpaca_prompt = """Below is an instruction that describes a task.
Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, input, output) + eos_token
        texts.append(text)
    return { "text" : texts}

In [27]:
from datasets import load_dataset
dataset = load_dataset("yahma/alpaca-cleaned", split = "train")

dataset = dataset.shuffle(seed=3407).select(range(1500))
# Apply the formatting function
dataset = dataset.map(formatting_prompts_func,
                      batched = True,
                      remove_columns=dataset.column_names #keeps only ttext
                      )

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

In [30]:
dataset

Dataset({
    features: ['text'],
    num_rows: 1500
})

In [33]:
dataset[0]

{'text': 'Below is an instruction that describes a task. \nWrite a response that appropriately completes the request.\n\n### Instruction:\nList 5 popular dishes in US.\n\n### Input:\n\n\n### Response:\n1. Hamburger: A classic American dish consisting of a beef patty served on a bun, often topped with cheese and various toppings such as lettuce, tomatoes, onions, and condiments like ketchup and mustard.\n\n2. Macaroni & Cheese: A comforting dish made from elbow macaroni mixed with a creamy cheese sauce and often baked until golden brown and bubbly.\n\n3. Fried Chicken: A southern staple, fried chicken is typically made from pieces of chicken that are coated in a seasoned batter and deep-fried until crispy and golden brown.\n\n4. Pizza: Pizza is a beloved food in the United States, with toppings ranging from classic pepperoni and cheese to more unique combinations like pineapple and ham.\n\n5. Apple Pie: A classic American dessert made from a flaky pastry crust filled with a sweet and ta

In [34]:
!pip install psutil

In [36]:
import psutil, time #lib to monitor system ram
torch.cuda.empty_cache() #release unused cache memory held by pytorch
torch.cuda.reset_peak_memory_stats() #allows to measure maximum vram consumed from this point forward

In [37]:
# before stats of fine tuning for performance benchmarking
process = psutil.Process() #current python interpretor process
train_start_time = time.time() #exact timestamp in sec
cpu_ram_before = process.memory_info().rss / 1024**3 #how much system ran (not GPU) is used before model loads data batches    (rss-> Resident Set Size in GB's)


In [38]:
process

psutil.Process(pid=9092, name='python3', status='running', started='19:12:47')

In [39]:
train_start_time

1774470158.6131253

In [40]:
cpu_ram_before

2.441661834716797

In [41]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_len,
    dataset_num_proc = 2,
    packing = True, #packs multiple short seq in 1 sequence
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # Set this to a higher number for a full run (e.g., 300)
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = 'none',
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/1500 [00:00<?, ? examples/s]

In [44]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,500 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 25,231,360 of 1,125,279,744 (2.24% trained)


Step,Training Loss
10,1.895300
20,1.818300
30,1.996600
40,1.842300
50,1.883400
60,1.926000


In [47]:
#  GPU Memory Stats
used_vram = round(torch.cuda.max_memory_reserved() / 1024**3, 3) # GB
used_memory_stats = torch.cuda.get_device_properties(0).total_memory / 1.074e+9
used_memory_percentage = round(used_vram / used_memory_stats * 100, 3)

# End timers and CPU RAM tracking
train_end_time = time.time()
train_duration = train_end_time - train_start_time # in seconds
cpu_ram_after = process.memory_info().rss / 1024**3
cpu_ram_used = cpu_ram_after - cpu_ram_before

# Performance Report
print(f"\n" + "="*30)
print(f"   TRAINING PERFORMANCE REPORT")
print(f"="*30)
print(f"Total Training Time:   {train_duration / 60:.2f} minutes")
print(f"Peak VRAM Reserved:    {used_vram} GB ({used_memory_percentage}%)")
print(f"System RAM Consumed:   {cpu_ram_used:.2f} GB")
print(f"Training Speed:        {trainer_stats.metrics['train_samples_per_second']:.2f} samples/sec")
print(f"Final Loss:            {trainer_stats.metrics['train_loss']:.4f}")
print(f"="*30)


   TRAINING PERFORMANCE REPORT
Total Training Time:   16.57 minutes
Peak VRAM Reserved:    3.32 GB (22.803%)
System RAM Consumed:   0.72 GB
Training Speed:        6.01 samples/sec
Final Loss:            1.8937


In [48]:
from unsloth.chat_templates import get_chat_template

# Enable 2x faster inference
FastLanguageModel.for_inference(model)

# Define the prompt (Same as training!)
instruction = "Complete the Fibonacci sequence provided in the input."
input_data = "1, 1, 2, 3, 5, 8,"

# Format using the Alpaca template
prompt = alpaca_prompt.format(instruction, input_data, "")

# Tokenize and move to GPU
inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")

In [49]:
from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)

output = model.generate(
    **inputs,
    streamer = text_streamer,
    max_new_tokens = 64,
    use_cache = True # Required for fast generation
)

<s> Below is an instruction that describes a task. 
Write a response that appropriately completes the request.

### Instruction:
Complete the Fibonacci sequence provided in the input.

### Input:
1, 1, 2, 3, 5, 8,

### Response:
1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 1


In [50]:
model.save_pretrained("lora_model_unsloth") # Saves the tiny adapter weights
tokenizer.save_pretrained("lora_model_unsloth")

('lora_model_unsloth/tokenizer_config.json',
 'lora_model_unsloth/special_tokens_map.json',
 'lora_model_unsloth/tokenizer.model',
 'lora_model_unsloth/added_tokens.json',
 'lora_model_unsloth/tokenizer.json')